# Day 5: Neural Networks & ANN (Keras)

MNIST warm-up, then a feed-forward churn propensity model with early stopping and a head-to-head comparison against the recorded Phase 1 SVM champion. Run from the repository root.


In [11]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib

SEED = 42
keras.utils.set_random_seed(SEED)
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data/Telco-Customer-Churn.csv").exists():
    BASE_DIR = BASE_DIR.parent
DATA_PATH = BASE_DIR / "data/Telco-Customer-Churn.csv"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "data/reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Repo:", BASE_DIR)


TensorFlow: 2.21.0
Repo: /Users/apple/Desktop/timepass/RetailMind


## 1. MNIST warm-up

Flatten each 28×28 image into 784 values, scale pixels to [0, 1], and classify one of ten digits with a hidden ReLU layer and softmax output. This is a compact example of dense layers, activations, loss, and the Keras training loop.


In [12]:
mnist = keras.datasets.mnist
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = mnist.load_data()
x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0

mnist_model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
mnist_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mnist_model.fit(x_train_mnist, y_train_mnist, validation_split=0.1, epochs=5, batch_size=128, verbose=2)
mnist_loss, mnist_accuracy = mnist_model.evaluate(x_test_mnist, y_test_mnist, verbose=0)
print({"MNIST_test_loss": round(float(mnist_loss), 4), "MNIST_test_accuracy": round(float(mnist_accuracy), 4)})


Epoch 1/5
422/422 - 1s - 2ms/step - accuracy: 0.8971 - loss: 0.3734 - val_accuracy: 0.9558 - val_loss: 0.1688
Epoch 2/5
422/422 - 0s - 905us/step - accuracy: 0.9506 - loss: 0.1710 - val_accuracy: 0.9658 - val_loss: 0.1256
Epoch 3/5
422/422 - 0s - 913us/step - accuracy: 0.9651 - loss: 0.1223 - val_accuracy: 0.9692 - val_loss: 0.1075
Epoch 4/5
422/422 - 0s - 1ms/step - accuracy: 0.9734 - loss: 0.0946 - val_accuracy: 0.9722 - val_loss: 0.0965
Epoch 5/5
422/422 - 0s - 900us/step - accuracy: 0.9786 - loss: 0.0761 - val_accuracy: 0.9733 - val_loss: 0.0897
{'MNIST_test_loss': 0.0961, 'MNIST_test_accuracy': 0.9701}


## 2. Prepare the Telco churn data

Keep the same stratified 80/20 split seed as Day 4. Create the same engineered features, fit scaling/category vocabularies on the training partition only, and apply those mappings to validation and test rows. The ANN receives numeric, normalized arrays.


In [16]:


training_data_path = BASE_DIR / "data" / "train_data.csv"
test_data_path = BASE_DIR / "data" / "test_data.csv"



train_data = pd.read_csv(training_data_path)
test_data = pd.read_csv(test_data_path)
xtrain = train_data.drop("Churn", axis=1)
ytrain = train_data["Churn"]
xtest = test_data.drop("Churn", axis=1)
ytest = test_data["Churn"]
print("Training:", xtrain.shape)
print("Testing:", xtest.shape)






# One-hot encode using training categories only; align held-out columns to training schema.
X_train = pd.get_dummies(xtrain, dummy_na=True, dtype="float32")
X_test = pd.get_dummies(xtest, dummy_na=True, dtype="float32").reindex(columns=X_train.columns, fill_value=0)
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)
means = X_train.mean()
stds = X_train.std().replace(0, 1).fillna(1)
X_train = ((X_train - means) / stds).to_numpy(dtype="float32")
X_test = ((X_test - means) / stds).to_numpy(dtype="float32")
y_train = ytrain.to_numpy(dtype="float32")
y_test = ytest.to_numpy(dtype="int32")
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.20, stratify=y_train, random_state=SEED)
print("Train / validation / test:", X_fit.shape, X_val.shape, X_test.shape)


Training: (5634, 19)
Testing: (1409, 19)
Train / validation / test: (4507, 60) (1127, 60) (1409, 60)


## 3. Train the ANN

A small 64→32 dense network keeps capacity modest for this tabular dataset. The validation set drives early stopping and restores the best validation-loss weights. The held-out test set is reserved for the final comparison.


In [17]:
ann = keras.Sequential([
    keras.layers.Input(shape=(X_fit.shape[1],)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.15),
    keras.layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc"), "accuracy"])
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
history = ann.fit(X_fit, y_fit, validation_data=(X_val, y_val), epochs=100, batch_size=64, callbacks=[early_stop], verbose=2)
ann.save(MODEL_DIR / "churn_ann_day5.keras")


Epoch 1/100
71/71 - 1s - 9ms/step - accuracy: 0.7526 - auc: 0.7642 - loss: 0.4901 - val_accuracy: 0.7977 - val_auc: 0.8271 - val_loss: 0.4355
Epoch 2/100
71/71 - 0s - 855us/step - accuracy: 0.7866 - auc: 0.8206 - loss: 0.4439 - val_accuracy: 0.8012 - val_auc: 0.8390 - val_loss: 0.4227
Epoch 3/100
71/71 - 0s - 819us/step - accuracy: 0.7874 - auc: 0.8321 - loss: 0.4315 - val_accuracy: 0.8066 - val_auc: 0.8418 - val_loss: 0.4197
Epoch 4/100
71/71 - 0s - 824us/step - accuracy: 0.7948 - auc: 0.8369 - loss: 0.4269 - val_accuracy: 0.8083 - val_auc: 0.8422 - val_loss: 0.4193
Epoch 5/100
71/71 - 0s - 831us/step - accuracy: 0.7934 - auc: 0.8389 - loss: 0.4239 - val_accuracy: 0.8119 - val_auc: 0.8439 - val_loss: 0.4171
Epoch 6/100
71/71 - 0s - 807us/step - accuracy: 0.7988 - auc: 0.8417 - loss: 0.4206 - val_accuracy: 0.8083 - val_auc: 0.8438 - val_loss: 0.4182
Epoch 7/100
71/71 - 0s - 811us/step - accuracy: 0.7970 - auc: 0.8414 - loss: 0.4213 - val_accuracy: 0.8101 - val_auc: 0.8436 - val_loss: 0

## 4. Compare against Phase 1

Use a consistent 0.5 decision threshold and identical test rows. The benchmark uses the saved Day 4 classical pipeline as the locally available persisted champion artifact; the README retains the recorded Day 3 SVM result and explains the split/feature mismatch in the historical report. If the Day 3 SVM artifact is restored, it can be scored here using its own documented feature schema.


In [19]:
def metrics(y_true, probabilities):
    pred = (probabilities >= 0.5).astype("int32")

    return {
        "Test Recall": recall_score(y_true, pred, zero_division=0),
        "Test Precision": precision_score(y_true, pred, zero_division=0),
        "Test F1": f1_score(y_true, pred, zero_division=0),
        "Test Accuracy": accuracy_score(y_true, pred),
        "Test ROC-AUC": roc_auc_score(y_true, probabilities),
    }


ann_prob = ann.predict(
    X_test,
    verbose=0
).ravel()

results = {
    "Day 5 Keras ANN": metrics(y_test, ann_prob)
}



champion_path = MODEL_DIR / "churn_v4.joblib"

if champion_path.exists():

    champion = joblib.load(champion_path)

    champion_prob = champion.predict_proba(xtest)[:, 1]

    results["Day 4 Classical Champion"] = metrics(
        ytest,
        champion_prob
    )

else:
    print("Day 4 champion model not found.")


# -------------------------
# Comparison table
# -------------------------

comparison = pd.DataFrame(results).T

comparison.index.name = "Model"

comparison.to_csv(
    REPORT_DIR / "day5_model_comparison.csv"
)

comparison

,Test Recall,Test Precision,Test F1,Test Accuracy,Test ROC-AUC
Model,,,,,
Day 5 Keras ANN,0.574866,0.619597,0.596394,0.793471,0.837764
Day 4 Classical Champion,0.625668,0.603093,0.614173,0.791341,0.833796


## Interpretation

Compare churn recall, F1, and ROC-AUC together. A deeper network is useful only if it improves the metric that matters on the same held-out rows without a large train/validation gap. For small structured datasets, boosted trees and linear models often remain stronger and simpler; neural networks become more attractive with larger datasets, unstructured inputs, or shared representations across tasks.
